# 🔬 Performance Comparison: MediaPipe vs Ball Tracking

This notebook compares the computational performance of two tracking methods:
1. **MediaPipe** (pose detection)
2. **Ball Tracking** (color-based detection)

**Metrics compared:** FPS, CPU Usage, Memory Usage

## Step 1: Install Required Libraries
Run this cell first (only needed once)

In [ ]:
# Install required packages (run once)
# Install the dependencies once from the repository root:
#   pip install -r requirements.txt

## Step 2: Import Libraries

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import time
import os
import psutil
import pandas as pd
from datetime import datetime

print("✅ All libraries imported successfully!")

## Step 3: Configure Your Video Path

⚠️ **IMPORTANT:** Change the path below to YOUR video file!

In [ ]:
# ========================================
# ⚠️ CHANGE THIS TO YOUR VIDEO PATH!
# ========================================

VIDEO_PATH = 'path/to/session.mkv'   # <- your video file

# How many frames to test (more = more accurate but slower)
MAX_FRAMES = 300

# Scale factor (smaller = faster processing)
SCALE_FACTOR = 0.6

print(f"Video path: {VIDEO_PATH}")
print(f"Max frames to process: {MAX_FRAMES}")

## Step 4: Define Helper Classes

In [ ]:
class PerformanceMonitor:
    """Monitors system performance metrics"""
    
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.metrics = {
            'frame_number': [],
            'processing_time_ms': [],
            'fps': [],
            'cpu_percent': [],
            'memory_mb': [],
            'method': []
        }
        self.start_time = None
        
    def start_frame(self):
        self.start_time = time.perf_counter()
        
    def end_frame(self, frame_num, method_name):
        end_time = time.perf_counter()
        processing_time = (end_time - self.start_time) * 1000
        
        cpu_percent = self.process.cpu_percent()
        memory_mb = self.process.memory_info().rss / (1024 * 1024)
        fps = 1000 / processing_time if processing_time > 0 else 0
        
        self.metrics['frame_number'].append(frame_num)
        self.metrics['processing_time_ms'].append(processing_time)
        self.metrics['fps'].append(fps)
        self.metrics['cpu_percent'].append(cpu_percent)
        self.metrics['memory_mb'].append(memory_mb)
        self.metrics['method'].append(method_name)
        
    def get_summary(self, method_name):
        indices = [i for i, m in enumerate(self.metrics['method']) if m == method_name]
        
        if not indices:
            return None
            
        fps_values = [self.metrics['fps'][i] for i in indices]
        cpu_values = [self.metrics['cpu_percent'][i] for i in indices]
        memory_values = [self.metrics['memory_mb'][i] for i in indices]
        time_values = [self.metrics['processing_time_ms'][i] for i in indices]
        
        return {
            'method': method_name,
            'frames_processed': len(indices),
            'fps_mean': np.mean(fps_values),
            'fps_std': np.std(fps_values),
            'cpu_mean': np.mean(cpu_values),
            'cpu_std': np.std(cpu_values),
            'memory_mean_mb': np.mean(memory_values),
            'memory_std_mb': np.std(memory_values),
            'processing_time_mean_ms': np.mean(time_values)
        }

print("✅ PerformanceMonitor class defined!")

In [ ]:
class MediaPipeTracker:
    """MediaPipe-based pose tracking"""
    
    def __init__(self):
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        
    def process_frame(self, frame):
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.pose.process(frame_rgb)
        return results.pose_landmarks is not None
    
    def close(self):
        self.pose.close()


class BallTracker:
    """Blue ball color-based tracking"""
    
    def __init__(self):
        self.lower_blue = np.array([85, 120, 150])
        self.upper_blue = np.array([135, 255, 255])
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        
    def process_frame(self, frame, roi=None):
        if roi is not None:
            x, y, w, h = roi
            frame_roi = frame[y:y+h, x:x+w]
        else:
            frame_roi = frame
        
        hsv = cv2.cvtColor(frame_roi, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_blue, self.upper_blue)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, self.kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, self.kernel)
        
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            largest = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest)
            if area > 400:
                return True
        return False
    
    def close(self):
        pass

print("✅ Tracker classes defined!")

## Step 5: Run the Comparison

⚠️ This will take a few minutes. Be patient!

In [ ]:
# Open video
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"❌ ERROR: Cannot open video at {VIDEO_PATH}")
    print("Please check the path and try again!")
else:
    # Get video info
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * SCALE_FACTOR)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * SCALE_FACTOR)
    
    print("=" * 60)
    print("VIDEO INFORMATION")
    print("=" * 60)
    print(f"Total frames: {total_frames}")
    print(f"Video FPS: {fps}")
    print(f"Resolution (scaled): {width}x{height}")
    print(f"Frames to process: {min(MAX_FRAMES, total_frames)}")
    print("=" * 60)
    
    cap.release()
    print("\n✅ Video opened successfully! Ready to run comparison.")

In [ ]:
# Run the comparison
print("🚀 Starting performance comparison...")
print("This will take a few minutes. Please wait...\n")

cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * SCALE_FACTOR)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * SCALE_FACTOR)

# Initialize
mediapipe_tracker = MediaPipeTracker()
ball_tracker = BallTracker()
monitor = PerformanceMonitor()

# ROI for ball tracking
blue_roi = [
    int(70 * SCALE_FACTOR), 
    int(330 * SCALE_FACTOR), 
    int(830 * SCALE_FACTOR), 
    int(625 * SCALE_FACTOR)
]

mediapipe_detections = 0
ball_detections = 0

# ============================================
# TEST 1: MediaPipe
# ============================================
print("--- Testing MediaPipe ---")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frames_processed = 0

while frames_processed < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.resize(frame, (width, height))
    
    monitor.start_frame()
    detected = mediapipe_tracker.process_frame(frame)
    monitor.end_frame(frames_processed, "MediaPipe")
    
    if detected:
        mediapipe_detections += 1
    
    frames_processed += 1
    
    if frames_processed % 50 == 0:
        print(f"  MediaPipe: {frames_processed}/{MAX_FRAMES} frames...")

print(f"  ✅ MediaPipe done! ({mediapipe_detections} detections)")

# ============================================
# TEST 2: Ball Tracking
# ============================================
print("\n--- Testing Ball Tracking ---")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frames_processed = 0

while frames_processed < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.resize(frame, (width, height))
    
    monitor.start_frame()
    detected = ball_tracker.process_frame(frame, roi=blue_roi)
    monitor.end_frame(frames_processed, "BallTracking")
    
    if detected:
        ball_detections += 1
    
    frames_processed += 1
    
    if frames_processed % 50 == 0:
        print(f"  Ball Tracking: {frames_processed}/{MAX_FRAMES} frames...")

print(f"  ✅ Ball Tracking done! ({ball_detections} detections)")

# Cleanup
cap.release()
mediapipe_tracker.close()

print("\n✅ Comparison complete!")

## Step 6: View Results 📊

In [ ]:
# Get summaries
mediapipe_summary = monitor.get_summary("MediaPipe")
ball_summary = monitor.get_summary("BallTracking")

print("=" * 70)
print("📊 PERFORMANCE COMPARISON RESULTS")
print("=" * 70)

print(f"\n{'Metric':<35} {'MediaPipe':<18} {'Ball Tracking':<18}")
print("-" * 70)

print(f"{'Frames Processed':<35} {mediapipe_summary['frames_processed']:<18} {ball_summary['frames_processed']:<18}")

print(f"{'FPS (mean ± std)':<35} {mediapipe_summary['fps_mean']:.1f} ± {mediapipe_summary['fps_std']:.1f}{'':>5} {ball_summary['fps_mean']:.1f} ± {ball_summary['fps_std']:.1f}")

print(f"{'CPU % (mean ± std)':<35} {mediapipe_summary['cpu_mean']:.1f} ± {mediapipe_summary['cpu_std']:.1f}{'':>5} {ball_summary['cpu_mean']:.1f} ± {ball_summary['cpu_std']:.1f}")

print(f"{'Memory MB (mean ± std)':<35} {mediapipe_summary['memory_mean_mb']:.1f} ± {mediapipe_summary['memory_std_mb']:.1f}{'':>5} {ball_summary['memory_mean_mb']:.1f} ± {ball_summary['memory_std_mb']:.1f}")

print(f"{'Processing Time (ms/frame)':<35} {mediapipe_summary['processing_time_mean_ms']:.2f}{'':>13} {ball_summary['processing_time_mean_ms']:.2f}")

mp_det_rate = (mediapipe_detections/mediapipe_summary['frames_processed'])*100
ball_det_rate = (ball_detections/ball_summary['frames_processed'])*100
print(f"{'Detection Rate %':<35} {mp_det_rate:.1f}{'':>15} {ball_det_rate:.1f}")

print("\n" + "=" * 70)

In [ ]:
# Analysis
fps_ratio = ball_summary['fps_mean'] / mediapipe_summary['fps_mean']

print("\n📈 ANALYSIS")
print("=" * 70)

if fps_ratio > 1:
    print(f"✅ Ball Tracking is {fps_ratio:.1f}x FASTER than MediaPipe")
else:
    print(f"✅ MediaPipe is {1/fps_ratio:.1f}x FASTER than Ball Tracking")

if mediapipe_summary['memory_mean_mb'] > ball_summary['memory_mean_mb']:
    diff = mediapipe_summary['memory_mean_mb'] - ball_summary['memory_mean_mb']
    print(f"✅ Ball Tracking uses {diff:.1f} MB LESS memory")
else:
    diff = ball_summary['memory_mean_mb'] - mediapipe_summary['memory_mean_mb']
    print(f"✅ MediaPipe uses {diff:.1f} MB LESS memory")

print("\n" + "=" * 70)
print("\n📝 RECOMMENDATION FOR THESIS:")
print("""
Ball Tracking:
  ✅ FASTER processing (good for real-time)
  ✅ LOWER memory usage
  ❌ Only tracks the ball, not body joints
  ❌ Sensitive to lighting conditions

MediaPipe:
  ✅ RICHER data (33 body landmarks)
  ✅ MORE ROBUST to lighting changes
  ❌ Higher computational cost
  ❌ Requires more memory

CONCLUSION: Use BOTH methods together for comprehensive analysis.
""")

## Step 7: Save Results to CSV (Optional)

In [ ]:
# Save to CSV
df = pd.DataFrame(monitor.metrics)
csv_filename = f"performance_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(csv_filename, index=False)

print(f"✅ Results saved to: {csv_filename}")
print(f"\nFile location: {os.path.abspath(csv_filename)}")

## 🎉 Done!

Results from this benchmark are reported in Table 4.2 of the thesis.